In [1]:
# !pip install -q arize-phoenix-client '../logger[Tracing]'
!pip install -q arize-phoenix-client 'arize[Tracing]'

In [2]:
!pip freeze | grep arize

arize==7.51.1
arize-phoenix-client==1.21.0


In [14]:
import os
# Cloud Instance
PHOENIX_API_KEY = os.environ["PHOENIX_API_KEY"]
PHOENIX_BASE_URL = "https://app.phoenix.arize.com/s/phoenix"
# PHOENIX_PROJECT_NAME = "demo_llama_index"
PHOENIX_PROJECT_NAME = "playground"

In [15]:
from phoenix.client import Client

# Cloud instance with API key
client = Client(
    base_url=PHOENIX_BASE_URL,
    api_key=PHOENIX_API_KEY,
)

In [16]:
# Get spans as pandas DataFrame for analysis
spans_df = client.spans.get_spans_dataframe(
    project_identifier=PHOENIX_PROJECT_NAME,
    limit=1000,
    # root_spans_only=True,  # Only get top-level spans
    # start_time=datetime.now() - timedelta(hours=24)
)
# spans_df = spans_df[:50]
spans_df

name span_kind parent_id  \
context.span_id                                        
312d73af8fac8937  ChatCompletion       LLM      None   
ab9162a57a80bc6b  ChatCompletion       LLM      None

# Spans

In [17]:
SPACE_ID = "U3BhY2U6NTA3MDpsTlIr"
PROJECT_NAME = "test-sdkv7-10-27-25-a"

if SPACE_ID == "SPACE_ID" or API_KEY == "API_KEY":
    raise ValueError("❌ NEED TO CHANGE SPACE AND/OR API_KEY")
else:
    print(
        "✅ Import and Setup Arize Client Done! Now we can start using Arize!"
    )

from arize.pandas.logger import Client

arize_client = Client(space_id=SPACE_ID, api_key=API_KEY)

✅ Import and Setup Arize Client Done! Now we can start using Arize!


In [18]:
# MODEL_VERSION = "test-version"
MODEL_VERSION = None

response = arize_client.log_spans(
    dataframe=spans_df,
    # evals_dataframe=evals_df, # if you do not have any evals, remove this line
    model_id=PROJECT_NAME,
    model_version=MODEL_VERSION,  # optional
)

# If successful, the server will return a status_code of 200
if response.status_code != 200:
    print(
        f"❌ logging failed with response code {response.status_code}, {response.text}"
    )
else:
    print("✅ You have successfully logged training set to Arize")

✅ You have successfully logged training set to Arize


# Evals

In [25]:
import pandas as pd

SPAN_ID = "312d73af8fac8937"  # demo_llama_index
# SPAN_ID = "26eb0c05014dcdb4" # playground
evaluation_data = {
    "context.span_id": [SPAN_ID],  # Use your span_id
    "eval.myeval.label": ["accuracy"],  # Example label name
    "eval.myeval.score": [0.95],  # Example label value
    "eval.myeval.explanation": ["some explanation"],
}
evals_df = pd.DataFrame(evaluation_data)
evals_df

   context.span_id eval.myeval.label  eval.myeval.score  \
0  312d73af8fac8937          accuracy               0.95  


In [26]:
# send the eval to Arize
arize_client.log_evaluations_sync(
    evals_df,
    PROJECT_NAME,
    MODEL_VERSION,
)

records_updated: 1

# Annotations

In [31]:
annotation_data = {
    "context.span_id": [SPAN_ID],
    "annotation.quality.label": ["good"],
    "annotation.relevance.label": ["relevant"],
    "annotation.relevance.updated_by": ["human_annotator_1"],
    "annotation.sentiment_score.score": [4.5],
    "annotation.notes": ["User confirmed the summary was helpful."],
}
annotations_df = pd.DataFrame(annotation_data)

In [33]:
response = arize_client.log_annotations(
    dataframe=annotations_df,
    project_name=PROJECT_NAME,
    validate=True,
    verbose=True,
)
# response

# Metadata

In [29]:
metadata_df = pd.DataFrame(
    {
        "context.span_id": [SPAN_ID],
        "attributes.metadata.status": ["reviewed"],
        "attributes.metadata.tag": ["important"],
    }
)

In [30]:
response = arize_client.log_metadata(
    dataframe=metadata_df,
    project_name=PROJECT_NAME,
    validate=True,
    verbose=True,
)

response

{'spans_processed': 1, 'spans_updated': 1, 'spans_failed': 0, 'errors': []}